# Makemore Part 3: Batch Normalization

In this notebook we train a deeper character-level language model on the `names.txt` dataset and add **Batch Normalization** to keep optimization stable. We also refactor the messy tensor code into clean, reusable layer classes that mirror the PyTorch `nn.Module` API.

## Learning objectives

- Understand why initialization scale matters and how Kaiming (He) init keeps activations stable.
- See how Batch Normalization controls the distribution of hidden activations and speeds up training.
- Train a multi-layer MLP for next-character prediction using a sliding-window dataset.
- Build reusable `Linear`, `BatchNorm1d`, and `Tanh` layers from scratch and compare them to `nnzero.layers`.
- Inspect activation/gradient distributions to diagnose training health.

## Why this matters

Deep networks are hard to train because activations and gradients can explode or vanish as they propagate through many layers. Batch Normalization is one of the most important practical techniques for making deep networks train reliably: it normalizes layer inputs to a standard scale and lets us use higher learning rates. Understanding *why* it works—and how to implement it yourself—is a key step from "copying PyTorch" to "reasoning about neural nets".

## Prerequisites

- The previous makemore notebooks (bigram and MLP baselines).
- Basic PyTorch tensors, `F.cross_entropy`, and `.backward()`.
- Familiarity with the micrograd `Value` class is helpful but not required here.


In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from nnzero import Linear, BatchNorm1d, Tanh, Module, Sequential
from nnzero.utils import build_vocab, build_dataset, split_dataset, load_names, set_seed, sample_from_model
%matplotlib inline

## Quick-run mode for CI

When the environment variable `NNZERO_QUICK_RUN=1` is set, training loops run for only a small number of steps so that notebooks can be validated quickly. Students should leave this unset to train the full models.

In [ ]:
import os
QUICK_RUN = os.environ.get("NNZERO_QUICK_RUN", "0") == "1"
if QUICK_RUN:
    print("Quick-run mode enabled: training loops will use far fewer steps.")

## Reproducibility

Deep-learning experiments involve randomness: weight initialization, data shuffling, and sampling all depend on RNG state. Fixing the seed makes the notebook deterministic, so you (and your instructor) can compare results across runs. `nnzero.utils.set_seed` seeds Python's `random` module and returns a dedicated PyTorch `Generator` so we don't pollute the global RNG.

In [ ]:
g = set_seed(2147483647)

## Load the dataset

`nnzero.utils.load_names` reads newline-separated words from a file. The path is given relative to the repository root (`data/names.txt`).

In [ ]:
words = load_names('data/names.txt')
words[:8]

In [ ]:
len(words)

## Build the vocabulary

Character-level models need a mapping from characters to integer indices. We use `.` as a special start/end token with index 0. `nnzero.utils.build_vocab` builds the `(stoi, itos, vocab_size)` triple for us, but the logic is simple enough to write by hand if you prefer.

> **Try it yourself:** look at `nnzero.utils.build_vocab` (or re-implement it) and verify that `.` gets index 0 and the remaining characters are sorted alphabetically.

In [ ]:
stoi, itos, vocab_size = build_vocab(words)
print("Vocab size:", vocab_size)
print("Index to char:", itos)

## Build the dataset

A character-level language model predicts the next character given a context of previous characters. We use a sliding window of length `block_size` over each word. `nnzero.utils.split_dataset` shuffles and splits the word list into train/validation/test sets, and `nnzero.utils.build_dataset` turns each split into `(X, Y)` tensors.

**Why this matters:** the train/val/test split lets us detect overfitting. If train loss keeps dropping but validation loss rises, the model is memorizing the training names instead of learning general name-structure patterns.

In [ ]:
block_size = 3  # context length: how many characters do we take to predict the next one?

train_words, val_words, test_words = split_dataset(words, shuffle=True, seed=42)

Xtr, Ytr = build_dataset(train_words, stoi, block_size=block_size)
Xdev, Ydev = build_dataset(val_words, stoi, block_size=block_size)
Xte, Yte = build_dataset(test_words, stoi, block_size=block_size)

print('train:', Xtr.shape, Ytr.shape)
print('val:  ', Xdev.shape, Ydev.shape)
print('test: ', Xte.shape, Yte.shape)

## MLP revisited: Kaiming init + BatchNorm

We train a one-hidden-layer MLP first, then scale up. Three details make modern deep networks trainable:

1. **Embedding lookup (`C[Xb]`):** each character index is mapped to a learned vector. This is far more expressive than one-hot encoding because similar characters can end up with similar embeddings.
2. **Kaiming initialization:** weights are drawn from $\mathcal{N}(0, 1/\text{fan\_in})$ and then scaled by a gain (here `5/3` for `tanh`). Without this, the pre-activations in deep networks would have huge variance, pushing `tanh` into its flat saturated region and killing gradients.
3. **Batch Normalization:** before the non-linearity we subtract the batch mean and divide by the batch standard deviation, then apply learned scale (`bngain`) and shift (`bnbias`). This keeps the distribution feeding `tanh` roughly fixed throughout training, which dramatically reduces sensitivity to initialization and allows larger learning rates.

> **Note:** The manual tensor implementation below is identical in spirit to `nnzero.layers.Linear` and `nnzero.layers.BatchNorm1d`. We write it out explicitly so you can see every operation; later we switch to reusable layer classes.

In [ ]:
# MLP revisited
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5) #* 0.2
#b1 = torch.randn(n_hidden,                        generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.01
b2 = torch.randn(vocab_size,                      generator=g) * 0

# BatchNorm parameters
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

## Training loop

We minimize the cross-entropy loss between predicted logits and the true next character. Cross-entropy is the natural choice for classification: it penalizes confident wrong predictions heavily and is mathematically equivalent to maximizing the log-likelihood of the training data.

The loop below also maintains **running estimates** of the batch-normalization mean and standard deviation. These running stats will replace the mini-batch stats at evaluation time, so inference does not depend on the batch size.

In [ ]:
# same optimization as last time
max_steps = 1000 if QUICK_RUN else 200000
batch_size = 32
lossi = []

for i in range(max_steps):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
  
  # forward pass
  emb = C[Xb] # embed the characters into vectors
  embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
  # Linear layer
  hpreact = embcat @ W1 #+ b1 # hidden layer pre-activation
  # BatchNorm layer
  # -------------------------------------------------------------
  bnmeani = hpreact.mean(0, keepdim=True)
  bnstdi = hpreact.std(0, keepdim=True)
  hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
  with torch.no_grad():
    bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
    bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi
  # -------------------------------------------------------------
  # Non-linearity
  h = torch.tanh(hpreact) # hidden layer
  logits = h @ W2 + b2 # output layer
  loss = F.cross_entropy(logits, Yb) # loss function
  
  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())
  

In [ ]:
plt.plot(lossi)

### 🧪 Try it yourself: initialization and BatchNorm

In the cell above we used Kaiming scaling `(5/3) / sqrt(fan_in)` and BatchNorm. What happens if you change one of them?

1. Remove the `5/3` gain from `W1` (use only `1 / sqrt(fan_in)`).
2. Or comment out the BatchNorm block and add back a small bias `b1`.

Retrain for a few thousand steps and compare the loss curve and final train/val loss.

In [ ]:
# Your experiment here


### Solution

In [ ]:
# Example: disable BatchNorm and use a small bias instead.
# You should see the loss start higher or plateau earlier because tanh saturates.

# g = set_seed(2147483647)
# C  = torch.randn((vocab_size, n_embd), generator=g)
# W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (1.0)/((n_embd * block_size)**0.5)
# b1 = torch.randn(n_hidden, generator=g) * 0.01
# W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
# b2 = torch.randn(vocab_size, generator=g) * 0
# parameters = [C, W1, b1, W2, b2]
# for p in parameters: p.requires_grad = True
# ... then run a shortened training loop and compare ...

## Calibrate BatchNorm at the end of training

During training, BatchNorm uses the statistics of the current mini-batch. For evaluation we want deterministic outputs that do not depend on batch composition. A common approach is to keep an exponential moving average of the training-batch statistics; alternatively, at the end of training we can pass the full training set through the network once and record the exact mean and standard deviation.

In [ ]:
# calibrate the batch norm at the end of training

with torch.no_grad():
  # pass the training set through
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 # + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnstd = hpreact.std(0, keepdim=True)


In [ ]:
@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 # + b1
  #hpreact = bngain * (hpreact - hpreact.mean(0, keepdim=True)) / hpreact.std(0, keepdim=True) + bnbias
  hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

## loss log

### original:
train 2.1245384216308594
val   2.168196439743042

### fix softmax confidently wrong:
train 2.07
val   2.13

### fix tanh layer too saturated at init:
train 2.0355966091156006
val   2.1026785373687744

### use semi-principled "kaiming init" instead of hacky init:
train 2.0376641750335693
val   2.106989622116089

### add batch norm layer
train 2.0668270587921143
val 2.104844808578491


## Summary + PyTorch-style layers

So far we wrote every tensor operation by hand. That is great for learning, but real models have dozens of layers. The next section introduces tiny `Linear`, `BatchNorm1d`, and `Tanh` classes that follow the same API as PyTorch modules (`__call__` for the forward pass, `parameters()` for optimization).

> **These classes also live in `nnzero.layers`.** We define them below so you can read the implementation, but in your own projects you can simply import `from nnzero import Linear, BatchNorm1d, Tanh, Sequential`.

In [ ]:
# Let's train a deeper network
# The classes we create here are the same API as nn.Module in PyTorch

class Linear:
  
  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in**0.5
    self.bias = torch.zeros(fan_out) if bias else None
  
  def __call__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out
  
  def parameters(self):
    return [self.weight] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
  
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    # buffers (trained with a running 'momentum update')
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)
  
  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      xmean = x.mean(0, keepdim=True) # batch mean
      xvar = x.var(0, keepdim=True) # batch variance
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    # update the buffers
    if self.training:
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []

n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 100 # the number of neurons in the hidden layer of the MLP
g = torch.Generator().manual_seed(2147483647) # for reproducibility

C = torch.randn((vocab_size, n_embd),            generator=g)
layers = [
  Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(           n_hidden, vocab_size, bias=False), BatchNorm1d(vocab_size),
]
# layers = [
#   Linear(n_embd * block_size, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden), Tanh(),
#   Linear(           n_hidden, vocab_size),
# ]

with torch.no_grad():
  # last layer: make less confident
  layers[-1].gamma *= 0.1
  #layers[-1].weight *= 0.1
  # all other layers: apply gain
  for layer in layers[:-1]:
    if isinstance(layer, Linear):
      layer.weight *= 1.0 #5/3

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

## Train a deeper network

Now we stack six layers. Without BatchNorm, a network this deep would be very hard to train: each layer multiplies the activation scale, so early layers would quickly saturate `tanh` and gradients would vanish. BatchNorm fixes the scale at every layer, making deep stacks behave like shallow ones during optimization.

The `ud` list tracks the ratio `std(update) / std(parameter)` for each parameter matrix. Ratios around `1e-3` are a useful rule of thumb that all layers are learning at a similar speed.

In [ ]:
# same optimization as last time
max_steps = 1000 if QUICK_RUN else 200000
batch_size = 32
lossi = []
ud = []

for i in range(max_steps):
  
  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y
  
  # forward pass
  emb = C[Xb] # embed the characters into vectors
  x = emb.view(emb.shape[0], -1) # concatenate the vectors
  for layer in layers:
    x = layer(x)
  loss = F.cross_entropy(x, Yb) # loss function
  
  # backward pass
  for layer in layers:
    layer.out.retain_grad() # AFTER_DEBUG: would take out retain_graph
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # update
  lr = 0.1 if i < 150000 else 0.01 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())
  with torch.no_grad():
    ud.append([((lr*p.grad).std() / p.data.std()).log10().item() for p in parameters])

  if i >= 1000:
    break # AFTER_DEBUG: would take out obviously to run full optimization

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out
    print('layer %d (%10s): mean %+.2f, std %.2f, saturated: %.2f%%' % (i, layer.__class__.__name__, t.mean(), t.std(), (t.abs() > 0.97).float().mean()*100))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('activation distribution')

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i, layer in enumerate(layers[:-1]): # note: exclude the output layer
  if isinstance(layer, Tanh):
    t = layer.out.grad
    print('layer %d (%10s): mean %+f, std %e' % (i, layer.__class__.__name__, t.mean(), t.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'layer {i} ({layer.__class__.__name__}')
plt.legend(legends);
plt.title('gradient distribution')

In [ ]:
# visualize histograms
plt.figure(figsize=(20, 4)) # width and height of the plot
legends = []
for i,p in enumerate(parameters):
  t = p.grad
  if p.ndim == 2:
    print('weight %10s | mean %+f | std %e | grad:data ratio %e' % (tuple(p.shape), t.mean(), t.std(), t.std() / p.std()))
    hy, hx = torch.histogram(t, density=True)
    plt.plot(hx[:-1].detach(), hy.detach())
    legends.append(f'{i} {tuple(p.shape)}')
plt.legend(legends)
plt.title('weights gradient distribution');

In [ ]:
plt.figure(figsize=(20, 4))
legends = []
for i,p in enumerate(parameters):
  if p.ndim == 2:
    plt.plot([ud[j][i] for j in range(len(ud))])
    legends.append('param %d' % i)
plt.plot([0, len(ud)], [-3, -3], 'k') # these ratios should be ~1e-3, indicate on plot
plt.legend(legends);


### 🧪 Try it yourself: depth and activation statistics

The histograms above reveal how activations and gradients flow through the network. Experiment with the architecture:

1. Add or remove one `Linear + BatchNorm1d + Tanh` block.
2. Or change the `1.0` gain applied to hidden `Linear` weights.
3. Re-run the training and visualization cells. How do saturation percentage, gradient std, and update ratios change?

In [ ]:
# Your experiment here


### Solution

In [ ]:
# Example: remove one hidden block and re-run.
# Fewer layers usually mean less saturation risk, but also a smaller representational budget.
# Compare the final validation loss to decide whether depth helps on this dataset.

# layers = [
#   Linear(n_embd * block_size, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
#   Linear(           n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
#   Linear(           n_hidden, vocab_size, bias=False), BatchNorm1d(vocab_size),
# ]
# ... re-run cells that define model, train, and plot ...

In [ ]:
@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  x = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  for layer in layers:
    x = layer(x)
  loss = F.cross_entropy(x, y)
  print(split, loss.item())

# put layers into eval mode
for layer in layers:
  layer.training = False
split_loss('train')
split_loss('val')

## Sample from the model

To generate a new name we feed the model a context of `.` tokens and repeatedly sample the next character from the predicted probability distribution. `nnzero.utils.sample_from_model` wraps exactly this loop, but you could also write it out explicitly as in the earlier notebooks.

In [ ]:
# sample_from_model expects a callable that takes (1, block_size) character indices
# and returns logits. Our `layers` list starts after the embedding lookup, so we
# wrap the embedding table and the MLP layers into one object.
class CharMLP:
    def __init__(self, C, layers):
        self.C = C
        self.layers = layers

    def __call__(self, x):
        emb = self.C[x]                 # (B, T, n_embd)
        x = emb.view(emb.shape[0], -1)  # (B, T * n_embd)
        for layer in self.layers:
            x = layer(x)
        return x

model = CharMLP(C, layers)
for layer in model.layers:
    if hasattr(layer, 'training'):
        layer.training = False  # BatchNorm must use running stats at inference time

g = torch.Generator().manual_seed(2147483647 + 10)
samples = sample_from_model(model, stoi, itos, block_size, generator=g, n_samples=20)
for s in samples:
    print(s)


## Bonus content (not covered in the video)

The cells below are optional interactive experiments that build intuition for normalization and the scale of activations/gradients in linear layers.

### BatchNorm as a normalization widget

Interact with the slider to see how a single input value changes the mean and standard deviation of a mini-batch, and therefore shifts the normalized outputs of every example. This is why BatchNorm couples examples within a batch during training but becomes deterministic at evaluation time.

In [ ]:
# BatchNorm forward pass as a widget

from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
import scipy.stats as stats
import numpy as np

def normshow(x0):
  
  g = torch.Generator().manual_seed(2147483647+1)
  x = torch.randn(5, generator=g) * 5
  x[0] = x0 # override the 0th example with the slider
  mu = x.mean()
  sig = x.std()
  y = (x - mu)/sig

  plt.figure(figsize=(10, 5))
  # plot 0
  plt.plot([-6,6], [0,0], 'k')
  # plot the mean and std
  xx = np.linspace(-6, 6, 100)
  plt.plot(xx, stats.norm.pdf(xx, mu, sig), 'b')
  xx = np.linspace(-6, 6, 100)
  plt.plot(xx, stats.norm.pdf(xx, 0, 1), 'r')
  # plot little lines connecting input and output
  for i in range(len(x)):
    plt.plot([x[i],y[i]], [1, 0], 'k', alpha=0.2)
  # plot the input and output values
  plt.scatter(x.data, torch.ones_like(x).data, c='b', s=100)
  plt.scatter(y.data, torch.zeros_like(y).data, c='r', s=100)
  plt.xlim(-6, 6)
  # title
  plt.title('input mu %.2f std %.2f' % (mu, sig))

interact(normshow, x0=(-30,30,0.5));


### Why Kaiming init: forward and backward statistics

For a linear layer `c = b @ a`, the standard deviation of the output is roughly `std(a) * std(b) * sqrt(fan_in)`. To keep `std(c) ≈ 1` we need `std(b) ≈ 1 / sqrt(fan_in)`. The same scaling keeps the backward gradients from exploding or vanishing. Run the experiment below with and without the ` / n**0.5` scaling to see the difference.

In [ ]:
# Linear: activation statistics of forward and backward pass

g = torch.Generator().manual_seed(2147483647)

a = torch.randn((1000,1), requires_grad=True, generator=g)          # a.grad = b.T @ c.grad
b = torch.randn((1000,1000), requires_grad=True, generator=g)       # b.grad = c.grad @ a.T
c = b @ a
loss = torch.randn(1000, generator=g) @ c
a.retain_grad()
b.retain_grad()
c.retain_grad()
loss.backward()
print('a std:', a.std().item())
print('b std:', b.std().item())
print('c std:', c.std().item())
print('-----')
print('c grad std:', c.grad.std().item())
print('a grad std:', a.grad.std().item())
print('b grad std:', b.grad.std().item())

### Linear + BatchNorm statistics

BatchNorm explicitly normalizes the output of the linear layer to unit variance before the next operation. Notice that the gradient statistics stay well behaved even when the linear weights are initialized with larger variance.

In [ ]:
# Linear + BatchNorm: activation statistics of forward and backward pass

g = torch.Generator().manual_seed(2147483647)

n = 1000
# linear layer ---
inp = torch.randn(n, requires_grad=True, generator=g)
w = torch.randn((n, n), requires_grad=True, generator=g) # / n**0.5
x = w @ inp
# bn layer ---
xmean = x.mean()
xvar = x.var()
out = (x - xmean) / torch.sqrt(xvar + 1e-5)
# ----
loss = out @ torch.randn(n, generator=g)
inp.retain_grad()
x.retain_grad()
w.retain_grad()
out.retain_grad()
loss.backward()

print('inp std: ', inp.std().item())
print('w std: ', w.std().item())
print('x std: ', x.std().item())
print('out std: ', out.std().item())
print('------')
print('out grad std: ', out.grad.std().item())
print('x grad std: ', x.grad.std().item())
print('w grad std: ', w.grad.std().item())
print('inp grad std: ', inp.grad.std().item())